# Phase 1: Multi-View Inference Baseline (Stages 0, 1, 2)

**Objective:** Run multi-view 3D face geometry reconstruction across 5 test subjects to verify that:
1. Identity shape regression isolates distinct individuals ($\|\beta_a - \beta_b\| > 10^{-3}$, eliminating mean-face fallback).
2. Multi-view fusion in embedding space ($w_i = \text{det\_score}_i \cdot \cos^2(\text{yaw}_i)$) handles pose variation without profile degradation.
3. Base mesh is strictly normalized to canonical neutral geometry ($\psi=0, \theta=0$) for UE5 blendshape compatibility.
4. Every subject directory outputs `head_mesh.obj`, `head_mesh.png` (Lambertian shaded render), and `manifest.json`.

In [ ]:
# ── CELL 1: Environment Check & Safe Kernel Restart for NumPy ABI Compatibility ──
import sys
import subprocess
import os
import numpy as np

print(f"[Setup] Active NumPy version: {np.__version__}")
if np.__version__.startswith("2."):
    print("Detected NumPy 2.x. Downgrading to 1.26.4 for C-extension ABI stability...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "--quiet"])
    print("Restarting kernel...")
    os.kill(os.getpid(), 9)
else:
    print("NumPy version is compatible. Proceeding.")

In [ ]:
# ── CELL 2: Install Repository Dependencies & GPU Extensions ────────────────────
!pip install -r requirements.txt --quiet
!pip install git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation --quiet

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

In [ ]:
# ── CELL 3: Path Setup & Model Asset Verification ───────────────────────────────
from pathlib import Path

# Ensure repo is on sys.path
import sys
sys.path.insert(0, '.')

from src.pipeline import FaceGeoPipeline
from evaluation.identity_score import compute_identity_score, render_neutral_preview

MODEL_DIR = Path("./models_cache")
CONFIG_PATH = Path("./configs/default.yaml")

print("Verifying model assets...")
print(f"Config exists: {CONFIG_PATH.exists()}")
print(f"Model cache dir: {MODEL_DIR.resolve()}")

In [ ]:
# ── CELL 4: Initialize Face Geometry Pipeline ──────────────────────────────────
pipeline = FaceGeoPipeline(
    config_path=str(CONFIG_PATH),
    model_dir=str(MODEL_DIR)
)
print("FaceGeoPipeline initialized successfully.")

In [ ]:
# ── CELL 5: Multi-Subject Inference Across 5 Subjects ──────────────────────────
# Expected structure:
# data/test_subjects/
# ├── subject_01/ [front.jpg, left.jpg, right.jpg]
# ├── subject_02/ ...
# └── subject_05/ ...

test_dir = Path("./data/test_subjects")
output_base = Path("./outputs/phase1_baseline")
output_base.mkdir(parents=True, exist_ok=True)

subject_dirs = sorted([d for d in test_dir.iterdir() if d.is_dir()]) if test_dir.exists() else []
results = {}
subject_betas = {}

if not subject_dirs:
    print(f"No test subject folders found in {test_dir}. Please supply 5 subject portrait folders.")
else:
    for s_dir in subject_dirs[:5]:
        subj_name = s_dir.name
        photos = sorted(list(s_dir.glob("*.jpg")) + list(s_dir.glob("*.png")))
        out_dir = output_base / subj_name
        print(f"\nProcessing {subj_name} ({len(photos)} photos)...")
        res = pipeline.run([str(p) for p in photos], str(out_dir))
        results[subj_name] = res

        # Load manifest to extract regressed beta shape vector
        import json
        with open(res['manifest_path']) as f:
            manifest = json.load(f)
        subject_betas[subj_name] = np.array(manifest['beta_shape'], dtype=np.float32)
        print(f"  Generated: {res['obj_path']}")
        print(f"  Preview: {res['preview_path']}")

In [ ]:
# ── CELL 6: Phase 1 Gate Verification: Pairwise Identity Distance ──────────────
# Gate Criterion 1: ||beta_a - beta_b|| > 1e-3 across all distinct subject pairs

if len(subject_betas) >= 2:
    names = list(subject_betas.keys())
    print("\n--- Pairwise Beta Euclidean Distances ---")
    gate_passed = True
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            dist = float(np.linalg.norm(subject_betas[names[i]] - subject_betas[names[j]]))
            print(f"{names[i]} vs {names[j]}: ||beta_a - beta_b|| = {dist:.5f}")
            if dist < 1e-3:
                print(f"  FAIL: Pair {names[i]} and {names[j]} collapsed to identical shape!")
                gate_passed = False
    
    assert gate_passed, "Gate Failure: One or more distinct subjects collapsed to the mean face!"
    print("\nSUCCESS: All subject identities are distinct with ||beta_a - beta_b|| > 1e-3.")
else:
    print("Need at least 2 subjects processed to compute pairwise distance matrix.")

In [ ]:
# ── CELL 7: Preview Renders & Identity Retention Scoring ───────────────────────
import matplotlib.pyplot as plt
import cv2

for subj_name, res in results.items():
    preview_file = res.get('preview_path')
    if preview_file and Path(preview_file).exists():
        img = cv2.imread(preview_file)[:, :, ::-1]
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(f"{subj_name} - Shaded Preview")
        plt.axis('off')
        plt.show()
    else:
        print(f"Warning: No preview found for {subj_name}")